In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
#from ufl import TrialFunction, TestFunction, inner, dx
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

In [ ]:
def refine(knots: np.ndarray, p: int, n_times: int=1)->np.ndarray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots = np.asarray(knots)
    mult_left = np.searchsorted(knots, knots[0], side='right')
    mult_right = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left = max(0, p + 1 - mult_left)
    pad_right = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)
        

In [ ]:
p0, p1 = 2,2
knots1 = np.array([-4., 0.,4.,], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=5)
knots2 = knots1# np.array([0, 0,0, 1, 1., 1], dtype=np.float64)
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0, p1])

In [ ]:
hs.mesh.plot_cells()

In [ ]:
my_cells = hs.mesh.meshes[0].cells
x_coords = my_cells[:, 0, :]
y_coords = my_cells[:, 1, :] 

points = np.zeros((4 * len(my_cells), 2), dtype=np.float64)
points[::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # Bottom-left
points[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # Bottom-right
points[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # Top-left
points[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # Top-right
cells = np.arange(len(my_cells)*4, dtype=np.int32).reshape(-1, 4)
# points = hs.mesh.meshes[0].cells.reshape(-1, 2)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells, coordinate_element, points)

In [ ]:
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

plotter = pyvista.Plotter()
plotter.add_mesh(grid.shrink(.9), show_edges=False, color="lightblue")
plotter.view_xy()
plotter.show(jupyter_backend="static")
print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
f = dolfinx.fem.Function(V)
sigma = 0.9
f.interpolate(lambda x: 1./(np.pi*sigma**4)*(1.-0.5*((x[0]**2+x[1]**2)/sigma**2)) * np.exp(-(x[0]**2+x[1]**2)/(2.*sigma**2)))
a0 = ufl.inner(u, v) * ufl.dx
f0 = ufl.inner(f, v)*ufl.dx

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ffi = cffi.FFI()

In [ ]:
Vsize = V.element.space_dimension
Usize = (p0+1)*(p1+1)
Ssize = Usize

M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = (np.kron(M, M).T @ np.kron(S_inv, S_inv) ).astype(dtype)
C_array = np.array(hs.level_spaces[0].bezier_operators, dtype=dtype)

operator_shape = (Usize, Ssize)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)

Lx, Ly = np.max(knots1), np.max(knots2)
Nx, Ny, = len(np.unique(knots1))-1, len(np.unique(knots2))-1
dx, dy = Lx/Nx, Ly/Ny
# C_func.x.array[:] = 2.

c_values = C_func.x.array.reshape((-1, Usize, Ssize))
for local_idx, midpoint in enumerate(midpoints):
    x, y = midpoint[0], midpoint[1]

    i = int(x//dx)
    j = int(y//dy)
    i = min(max(i,0), Nx-1)
    j = min(max(j,0), Ny-1)
    global_structured_idx = j*Nx+i
    Ci = C_array[global_structured_idx]
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = Ci@T
C_func.x.scatter_forward()

In [ ]:
import basix

dofs_per_cell=(p0+1)*(p1+1)
num_cells = hs.mesh.meshes[0].nelems
# Which BSpline functions are active on each cell.
# e.g
# array([[ 0,  1,  2,  6,  7,  8, 12, 13, 14],
#       [ 1,  2,  3,  7,  8,  9, 13, 14, 15], ...]
cells_to_dofs = hs.level_spaces[0].cell_to_basis(np.arange(num_cells, dtype=np.int32)).astype(np.int32)

num_control_points = np.max(cells_to_dofs)+1
# Create custom index map to tell dolfinx that the cells are in fact connected
# Keeps track of which cells are held by which process in parallel.
# In serial, should look something like 0..36 if there are 36 control points.
my_index_map = dolfinx.common.IndexMap(disconnected_mesh.comm, num_control_points)

ufl_element = basix.ufl.element("P", "quadrilateral", p0)
dummy_space = dolfinx.fem.functionspace(disconnected_mesh, ufl_element)
# The layout tells us the number of dofs and where they are (e.g 1 per vertex, 1 per edge, 0 per face, etc).
# Here, we only care about the number of dofs, so that we allocate the 
# correct amount of memory for it.
# https://docs.fenicsproject.org/dolfinx/v0.10.0.post5/cpp/doxygen/d9/d69/classdolfinx_1_1fem_1_1FunctionSpace.html
# https://docs.fenicsproject.org/dolfinx/main/python/generated/dolfinx.fem.html#dolfinx.fem.DofMap
# This corresponds to the C++ class ElementDofLayout https://github.com/FEniCS/dolfinx/blob/main/cpp/dolfinx/fem/ElementDofLayout.cpp
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object
dof_indices = cells_to_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells+1, dtype=np.int32)*dofs_per_cell).astype(np.int32)
adj = dolfinx.cpp.graph.AdjacencyList_int32(dof_indices, offsets) # Basically the same as cells_to_dof

# Here, we tell dolfinx several things at once:
# element_layout tells it how many dofs per cell we are going to have
# my_index_map keeps track of which process owns each cell
# and adj tells it which functions are present on each cell and how the cells connect globally. 
dofmap = dolfinx.cpp.fem.DofMap(element_layout, #element_dof_layout
                                my_index_map,  #index_map
                                1, #index_map_bs
                                adj, #AdjacencyList
                                1) #bs (block size of bsr array)
# Once this object is constructed, dolfinx knows how to populate the global array:
# once the kernel computation is bypassed, dolfinx receives a mxm array of floats, A_e.
# It then fetches dofmap, and for cell e, looks at the global/row indices for this cell and array,
# which returns a map/list of m global indices.
# Then, it loops through the array for local row i and col j, and adds it the following way:
# A_{glob}[map[i], map[j]] += A_e[i,j].

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    disconnected_mesh._cpp_object, 
    cpp_element, 
    dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    disconnected_mesh, 
    ufl_element, 
    V_spline_cpp
)
print(f"Success! Global B-Spline size: {V_spline.dofmap.index_map.size_global}")

In [ ]:
LOCAL_DOFS = (p0+1)*(p1+1)
@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    A = numba.carray(A_, (LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)

    # Get the operator (C_{B}^{BS} @ (C_{L}^{B}.T) @ S^{-1}) for this cell
    G = numba.carray(w_, (LOCAL_DOFS,LOCAL_DOFS), dtype=dtype)
    #B = B_flat.reshape((LOCAL_DOFS, LOCAL_DOFS))
    # Tabulate all sub blocks locally
    A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,# weights. Ignored for the L2 approximation because 
        # a0 does not depend on f and no value is looked up at the quadrature points
        # This is not very robust, one must be careful when calling this function
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    # Ci = C_array[entity_local_index]
    # B = Ci@T
    
    A[:, :] = G@A0@(G.T)

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_b(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array b_
    b = numba.carray(b_, (LOCAL_DOFS,), dtype=dtype)
    total_coeffs = LOCAL_DOFS+(LOCAL_DOFS**2)
    w_flat = numba.carray(w_, (total_coeffs), dtype=dtype)

    # The following line breaks in case we do not pass [f._cpp_object, C_func._cpp_object] in this
    # exact order or if f does not have the correct amount of dofs.
    B = w_flat[LOCAL_DOFS:].reshape((LOCAL_DOFS, LOCAL_DOFS))
    b0 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernelf0(ffi.from_buffer(b0),
        w_, # This line only works because we specified that b0 is of size LOCAL_DOFS,
        # which happens to also be the size of f._cpp_object
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )

    b[:] = B @ b0

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}
#coefficients_A = [C_func._cpp_object]
a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[
            V_spline._cpp_object, 
            V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[
                C_func._cpp_object
            ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0,1], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        [V_spline._cpp_object], # test space, determines the size of b_
        integrals_rhs, #give the adress of tabulate_b
        [f._cpp_object, C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        [], False, [], mesh=msh._cpp_object
    )
)

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
a_form = dolfinx.fem.form(a0)
A_cond = assemble_matrix(a_cond, bcs=[])
A_cond.assemble()
L_cond = assemble_vector(l_cond)
L_cond.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)


In [ ]:
ksp = PETSc.KSP().create(A_cond.comm)
ksp.setOperators(A_cond)
ksp.setType(PETSc.KSP.Type.CG)
ksp.getPC().setType(PETSc.PC.Type.JACOBI)
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(L_cond, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
f_sq_form = dolfinx.fem.form(ufl.inner(f, f) * ufl.dx)
f_sq_integral = dolfinx.fem.assemble_scalar(f_sq_form)
f_sq_integral = disconnected_mesh.comm.allreduce(f_sq_integral, op=MPI.SUM)
x_dot_L = L_cond.dot(u_sol.x.petsc_vec)
l2_error_sq = f_sq_integral - x_dot_L
l2_error = np.sqrt(abs(l2_error_sq))

print(f"Squared integral of f: {f_sq_integral}")
print(f"x^T L: {x_dot_L}")
print(f"L2 Error: {l2_error:.5e}")

In [ ]:
import basix
import cffi
import dolfinx
import numba
import numpy as np
import ufl
from dolfinx import default_real_type, default_scalar_type
from dolfinx.fem.petsc import assemble_matrix  # , assemble_vector
from dolfinx.jit import ffcx_jit
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature
from mpi4py import MPI

rtype = default_real_type
dtype = default_scalar_type


if __name__ == "__main__":
    print("Starting...")
    p0, p1 = 2, 2

    # Simple 2x1 disconnected mesh
    points = np.empty((8, 2), dtype=dtype)
    points[0, :] = [0, 0]
    points[1, :] = [1, 0]
    points[2, :] = [1, 1]
    points[3, :] = [0, 1]

    points[4, :] = [1, 0]
    points[5, :] = [2, 0]
    points[6, :] = [2, 1]
    points[7, :] = [1, 1]

    cells = np.arange(2 * 4, dtype=np.int32).reshape(-1, 4)

    coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
    disconnected_mesh = dolfinx.mesh.create_mesh(
        MPI.COMM_WORLD, cells, coordinate_element, points
    )

    legendre_elt = basix.ufl.element(
        "DG",
        "quadrilateral",
        degree=p0,
        lagrange_variant=basix.LagrangeVariant.legendre,
    )
    V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
    print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    f = dolfinx.fem.Function(V)
    f.interpolate(lambda x: np.sin(x[0]) * np.cos(x[1]))
    scalar_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
    scalar_func = dolfinx.fem.Function(scalar_space, dtype=dtype)
    scalar_func.x.array[:] = 1.0
    a0 = ufl.inner(u, v) * scalar_func * ufl.dx
    f0 = ufl.inner(f, v) * ufl.dx

    msh = disconnected_mesh
    ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
    kernela0 = getattr(
        ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}"
    )  # type: ignore

    ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
    kernelf0 = getattr(
        ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}"
    )  # type: ignore

    ffi = cffi.FFI()

    Vsize = V.element.space_dimension
    Usize = (p0 + 1) * (p1 + 1)
    Ssize = Usize

    operator_shape = ((p0 + 1) * (p1 + 1), (p0 + 1) * (p1 + 1))
    # Create a custom space that holds the content of each matrix for the relevant cell.
    # degree 0 because the value is constant over each cell
    C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
    C_func = dolfinx.fem.Function(C_space, dtype=dtype)
    C_func.x.array[:] = 2.0

    num_cells_local = disconnected_mesh.topology.index_map(
        disconnected_mesh.topology.dim
    ).size_local

    Nx = 2
    Ny = 1
    num_control_points = (Nx + p0) * (Ny + p1)
    # Create custom index map to tell dolfinx that the cells are in fact connected
    # Keeps track of which cells are held by which process in parallel.
    # In serial, should look something like 0..36 if there are 36 control points.
    my_index_map = dolfinx.common.IndexMap(disconnected_mesh.comm, num_control_points)

    dofs_per_cell = (p0 + 1) * (p1 + 1)
    num_cells = num_cells_local
    cells_to_dofs = (
        # hs.level_spaces[0].cell_to_basis(np.arange(num_cells)).astype(np.int32)
        np.array(
            [
                [0, 1, 2, 4, 5, 6, 8, 9, 10],
                [1, 2, 3, 5, 6, 7, 9, 10, 11],
            ],
            dtype=np.int32,
        )
    )

    ufl_element = basix.ufl.element("P", "quadrilateral", p0)
    dummy_space = dolfinx.fem.functionspace(disconnected_mesh, ufl_element)
    # The layout tells us the number of dofs and where they are (e.g 1 per vertex, 1 per edge, 0 per face, etc).
    # Here, we only care about the number of dofs, so that we allocate the
    # correct amount of memory for it.
    # https://docs.fenicsproject.org/dolfinx/v0.10.0.post5/cpp/doxygen/d9/d69/classdolfinx_1_1fem_1_1FunctionSpace.html
    # https://docs.fenicsproject.org/dolfinx/main/python/generated/dolfinx.fem.html#dolfinx.fem.DofMap
    # This corresponds to the C++ class ElementDofLayout https://github.com/FEniCS/dolfinx/blob/main/cpp/dolfinx/fem/ElementDofLayout.cpp
    element_layout = dummy_space.dofmap.dof_layout
    cpp_element = dummy_space.element._cpp_object
    dof_indices = cells_to_dofs.ravel().astype(np.int32)
    offsets = (np.arange(num_cells + 1, dtype=np.int32) * dofs_per_cell).astype(
        np.int32
    )
    adj = dolfinx.cpp.graph.AdjacencyList_int32(
        dof_indices, offsets
    )  # Basically the same as cells_to_dof

    dofmap = dolfinx.cpp.fem.DofMap(
        element_layout,  # element_dof_layout
        my_index_map,  # index_map
        1,  # index_map_bs
        adj,  # AdjacencyList
        1,
    )  # bs (block size of bsr array)

    V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
        disconnected_mesh._cpp_object, cpp_element, dofmap
    )

    V_spline = dolfinx.fem.FunctionSpace(disconnected_mesh, ufl_element, V_spline_cpp)
    print(f"Success! Global B-Spline size: {V_spline.dofmap.index_map.size_global}")

    LOCAL_DOFS = (p0 + 1) * (p1 + 1)

    @numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
    def tabulate_A(
        A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None
    ):
        A = numba.carray(A_, (LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
        B = numba.carray(w_, (LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)

        # Tabulate all sub blocks locally
        A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
        kernela0(
            ffi.from_buffer(A0),
            w_,
            c_,
            coords_,
            entity_local_index,
            permutation,
            empty_void_pointer(),
        )
        A[:, :] = B @ A0 @ (B.T)

    formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore

    # # Gets the number of cells for which each individual core is responsible for.
    cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local)
    integrals = {
        dolfinx.fem.IntegralType.cell: [
            (0, tabulate_A.address, cells, np.array([0, 1], dtype=np.int8))
        ]
    }
    a_cond = dolfinx.fem.Form(  # We are not forming anything yet, this is a recipe
        formtype(  # selectes the correct floating-point precision
            spaces=[
                V_spline._cpp_object,
                V_spline._cpp_object,
            ],  # trial and test spaces, determines the size of A_
            integrals=integrals,  # this is a dictionary, and we are passing the adress of tabulate_A() here
            coefficients=[
                C_func._cpp_object,
                scalar_func._cpp_object,
            ],  # weights w_, holds C@T
            constants=[],
            need_permutation_data=False,
            entity_maps=[],
            mesh=msh._cpp_object,
        )
    )
    a_form = dolfinx.fem.form(a0)

    A_cond = assemble_matrix(a_cond)
    A_cond.assemble()
    A_dense = A_cond.convert("dense")
    A_np = A_dense.getDenseArray()
    #print(A_np)

    import scipy.sparse as sps

# Extract the CSR (Compressed Sparse Row) arrays from PETSc
indptr, indices, data = A_cond.getValuesCSR()

# Get the global size of the matrix
shape = A_cond.getSize()

# Create a SciPy CSR matrix
A_scipy = sps.csr_array((data, indices, indptr), shape=shape)

print(f"Matrix shape: {A_scipy.shape}")
print(f"Number of non-zeros: {A_scipy.nnz}")

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
# plt.spy plots the non-zero entries of a matrix
plt.spy(A_scipy, markersize=20, color='blue')
plt.title("Sparsity Pattern of B-Spline Mass Matrix")
plt.show()